In [11]:
import pandas as pd
import sqlite3
import os

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

DB_PATH = os.path.join(BASE_DIR, "2_DATABASE", "eko_cards.db")
PRICES_PATH = os.path.join(BASE_DIR, "3_PROJECT_DATA", "prices.xlsx")


# =========================
# TRANSACTION IMPORT (TYPE 2)
# =========================
def transaction_import():

    file_path = os.path.join(BASE_DIR, "3_PROJECT_DATA", "eko_transactions.xlsx")

    data = pd.read_excel(file_path)
    data.columns = data.columns.str.strip()

    # -------------------------
    # RENAME (BG → EN)
    # -------------------------
    data = data.rename(columns={
        "Станция": "plant",
        "Име": "name",
        "Номер на карта": "card_number",
        "Име на артикул": "material",
        "Дата": "date",
        "Литри": "bill_qty",
        "Единична цена": "price",
        "Сума": "amount",
        "Одометър": "Km_stand"
    })

    # -------------------------
    # CLEAN MATERIAL
    # -------------------------
    data["material"] = (
        data["material"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.upper()
    )

    # -------------------------
    # CLEAN CARD NUMBER
    # -------------------------
    data["card_number"] = (
        data["card_number"]
        .astype(str)
        .str.replace(r"\D", "", regex=True)
    )

    data = data[data["card_number"].str.len() > 0]

    # -------------------------
    # DATE (contains time)
    # -------------------------
    data["date"] = pd.to_datetime(data["date"], dayfirst=True, errors="coerce")
    data = data[data["date"].notna()]

    data["auth_time"] = data["date"].dt.time
    data["date"] = data["date"].dt.strftime("%Y-%m-%d")

    # -------------------------
    # NUMERIC CLEAN
    # -------------------------
    data["bill_qty"] = pd.to_numeric(data["bill_qty"], errors="coerce")
    data = data[data["bill_qty"] > 0]

    data["price"] = pd.to_numeric(data["price"], errors="coerce")
    data["amount"] = pd.to_numeric(data["amount"], errors="coerce")

    # -------------------------
    # ADD MISSING COLUMNS
    # -------------------------
    data["bill_qty2"] = None

    # -------------------------
    # FINAL COLUMN SELECTION ❗ (FIX)
    # -------------------------
    data = data[[
        "plant",
        "name",
        "card_number",
        "material",
        "date",
        "bill_qty",
        "bill_qty2",
        "price",
        "amount",
        "auth_time",
        "Km_stand"
    ]]

    # -------------------------
    # DB
    # -------------------------
    conn = sqlite3.connect(DB_PATH)

    data.to_sql(
        "transactions",
        conn,
        if_exists="append",
        index=False
    )

    conn.close()

    print(f"Transactions imported: {len(data)} rows")


# =========================
# PRICES IMPORT
# =========================
def price_import():

    prices = pd.read_excel(PRICES_PATH)
    prices.columns = prices.columns.str.strip()

    prices = prices.rename(columns={
        "ДАТА": "date",
        "ФИРМА": "company",
        "ЕИК": "eik",
        "ПРОДУКТ": "product",
        "МАРЖ": "margin",
        "КРАЙНА_ЦЕНА": "final_price",
        "ОТСТЪПКА": "discount"
    })

    prices["product"] = prices["product"].astype(str).str.upper().str.strip()
    prices["company"] = prices["company"].astype(str).str.strip()
    prices["eik"] = prices["eik"].astype(str).str.strip()

    prices["date"] = pd.to_datetime(prices["date"], dayfirst=True, errors="coerce")
    prices = prices[prices["date"].notna()]
    prices["date"] = prices["date"].dt.strftime("%Y-%m-%d")

    prices["final_price"] = pd.to_numeric(prices["final_price"], errors="coerce")
    prices["margin"] = pd.to_numeric(prices["margin"], errors="coerce")
    prices["discount"] = pd.to_numeric(prices["discount"], errors="coerce").fillna(0)

    conn = sqlite3.connect(DB_PATH)

    prices.to_sql("prices", conn, if_exists="append", index=False)

    conn.close()

    print(f"Prices imported: {len(prices)} rows")


# =========================
# COMPANY IMPORT
# =========================
def company_import():

    cards = pd.read_excel("../3_PROJECT_DATA/cards.xlsx", dtype={"Number": str})

    cards = cards.rename(columns={
        "Company": "company",
        "Name": "vehicle",
        "Number": "card_number"
    })

    cards["card_number"] = cards["card_number"].astype(str)

    conn = sqlite3.connect(DB_PATH)

    cards.to_sql("cards", conn, if_exists="replace", index=False)

    conn.close()


# =========================
# MERGE
# =========================
def merge_tables():

    conn = sqlite3.connect(DB_PATH)

    query = """
    SELECT
        t.*,
        c.company,
        p.margin,
        p.discount,

        CASE
            WHEN COALESCE(p.discount, 0) > 0 THEN
                ROUND(
                    CASE
                        WHEN (t.price - p.discount) < 0 THEN 0
                        ELSE (t.price - p.discount)
                    END, 2
                )

            WHEN p.final_price IS NOT NULL THEN
                ROUND(p.final_price, 2)

            ELSE
                ROUND(t.price, 2)
        END AS final_price

    FROM transactions t

    LEFT JOIN cards c
        ON t.card_number = c.card_number

    LEFT JOIN prices p
        ON c.eik = p.eik
        AND UPPER(p.product) = UPPER(t.material)
        AND p.date = (
            SELECT MAX(p2.date)
            FROM prices p2
            WHERE p2.eik = c.eik
              AND UPPER(p2.product) = UPPER(t.material)
              AND p2.date <= t.date
        )
    """

    data = pd.read_sql(query, conn)
    conn.close()

    return data


# =========================
# RESULT
# =========================
def generate_result(data):

    os.makedirs(os.path.join(BASE_DIR, "A_FINAL_RESULT"), exist_ok=True)

    output_file = os.path.join(BASE_DIR, "A_FINAL_RESULT", "eko_transactions.xlsx")

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

        for company, df in data.groupby("company", dropna=False):

            df = df.copy()

            company_name = "Unknown" if pd.isna(company) else str(company)
            sheet_name = company_name[:31]

            # -------------------------
            # ENSURE COLUMNS EXIST
            # -------------------------
            required_cols = [
                "bill_qty2", "margin", "discount", "Km_stand"
            ]

            for col in required_cols:
                if col not in df.columns:
                    df[col] = 0

            # -------------------------
            # NUMERIC SAFETY
            # -------------------------
            df["price"] = pd.to_numeric(df["price"], errors="coerce").round(2)
            df["final_price"] = pd.to_numeric(df["final_price"], errors="coerce").round(2)
            df["bill_qty"] = pd.to_numeric(df["bill_qty"], errors="coerce").fillna(0)

            df["margin"] = pd.to_numeric(df["margin"], errors="coerce").fillna(0).round(3)
            df["discount"] = pd.to_numeric(df["discount"], errors="coerce").fillna(0).round(3)

            # -------------------------
            # TOTALS
            # -------------------------
            df["eko_total"] = (df["bill_qty"] * df["price"]).round(2)
            df["gta_total"] = (df["bill_qty"] * df["final_price"]).round(2)

            # -------------------------
            # PROFIT
            # -------------------------
            df["profit"] = (
                (df["price"] - df["final_price"]) * df["bill_qty"]
            ).round(2)

            # -------------------------
            # COLUMN ORDER
            # -------------------------
            df = df[
                [
                    "date",
                    "plant",
                    "name",
                    "card_number",
                    "material",
                    "bill_qty",
                    "bill_qty2",
                    "final_price",
                    "discount",
                    "margin",
                    "profit",
                    "gta_total",
                    "price",
                    "eko_total",
                    "auth_time",
                    "Km_stand"
                ]
            ]

            # -------------------------
            # RENAME
            # -------------------------
            df = df.rename(columns={
                "date": "Дата",
                "plant": "Станция",
                "name": "Име",
                "card_number": "Номер на карта",
                "material": "Име на артикул",
                "bill_qty": "Литри",
                "bill_qty2": "Тип количество",
                "final_price": "GTA цена",
                "discount": "Отстъпка",
                "margin": "Марж",
                "profit": "Печалба",
                "gta_total": "Сума по GTA цена",
                "price": "ЕКО цена",
                "eko_total": "Сума по ЕКО цена",
                "auth_time": "Час",
                "Km_stand": "Километри"
            })

            df.to_excel(writer, sheet_name=sheet_name, index=False)

    print("Output file generated:", output_file)


# =========================
# PIPELINE
# =========================
def execute_pipeline():

    conn = sqlite3.connect(DB_PATH)
    conn.execute("DELETE FROM transactions")
    conn.commit()
    conn.close()

    transaction_import()
    data = merge_tables()
    generate_result(data)

Transactions imported: 466 rows
Output file generated: C:\Users\sveto\OneDrive\Desktop\eko_inv\A_FINAL_RESULT\eko_transactions.xlsx
